# Install and Import Library

In [ ]:
# Menginstall Sastrawi, Wordcloud, IndoNLP, dan NLTK
!pip install Sastrawi wordcloud nltk indoNLP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 11.1 MB/s eta 0:00:00


In [ ]:
# Import Pandas & Regex (re)
import pandas as pd # untuk manipulasi dan pengolahan data
import re # untuk preprocessing teks

# Import NLTK untuk tokenisasi teks, & Sastrawi untuk stopword removal dan stemming bahasa Indonesia
import nltk
from nltk.tokenize import RegexpTokenizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Import IndoNLP untuk normalisasi teks (kata yang dilebihkan dan slang)
from indoNLP.preprocessing import replace_word_elongation, replace_slang

# Import Drive
from google.colab import drive  # menghubungkan Google Drive ke Colab

In [ ]:
# Mengunduh resource NLTK yang dibutuhkan untuk proses tokenisasi
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# Load Dataset

In [ ]:
# Menghubungkan dengan Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path ke file dataset di Google Drive
file_path = '/content/drive/MyDrive/skripsi/svm_indobert/data/final_data/krl_complaints_labeled_2.csv'

# Membaca dataset CSV ke dalam DataFrame
df = pd.read_csv(file_path)

# Menampilkan 5 baris pertama untuk memastikan data berhasil dimuat
print('Preview data:')
display(df.head())

# Menampilkan informasi struktur dataset
print('\nInformasi dataset:')
df.info()

Preview data:


,created_at,full_text,in_reply_to_screen_name,lang,tweet_url,label
0,Mon Feb 26 12:06:32 +0000 2024,@CommuterLine @suckislife_ @perkeretaapian Nge...,CommuterLine,in,https://x.com/undefined/status/176208674293990...,Keamanan
1,Tue Jul 11 01:05:30 +0000 2023,@CommuterLine 5517 telat 50 menit kenapa masuk...,CommuterLine,in,https://x.com/undefined/status/167857118281009...,Keterlambatan
2,Mon Feb 10 23:48:19 +0000 2025,@CommuterLine itu papan jadwal cisauk jalur 2 ...,CommuterLine,in,https://x.com/undefined/status/188909910630558...,Keterlambatan
3,Wed Nov 08 13:09:49 +0000 2023,@CommuterLine @jalur5_ Klo kyk gini petugasnya...,CommuterLine,in,https://x.com/undefined/status/172224000284600...,Pelayanan
4,Fri Dec 01 09:48:55 +0000 2023,@CommuterLine Penataran Malang-SBY pukul jam 3...,CommuterLine,in,https://x.com/undefined/status/173052436570254...,Keterlambatan



Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4924 entries, 0 to 4923
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   created_at               4924 non-null   object
 1   full_text                4924 non-null   object
 2   in_reply_to_screen_name  3329 non-null   object
 3   lang                     3329 non-null   object
 4   tweet_url                3329 non-null   object
 5   label                    4924 non-null   object
dtypes: object(6)
memory usage: 230.9+ KB


# Inisialisasi Preprocessing
berisi resource dan konfigurasi yang digunakan secara konsisten dalam pipeline preprocessing teks.

## Lexical Resources
berisi data linguistik seperti kamus dan stopword.

In [ ]:
# Kamus Singkatan Stasiun KRL di Indonesia
# untuk standarisasi singkatan kode stasiun yang kemungkinan digunakan pengguna
KAMUS_STASIUN = {
    # LINTAS BOGOR (CENTRAL & DEPOK LINE)
    'jakk': 'lokasi_stasiun', 'jakarta kota': 'lokasi_stasiun', 'jakartakota': 'lokasi_stasiun', 'jkt kota': 'lokasi_stasiun', 'beos': 'lokasi_stasiun',
    'jay': 'lokasi_stasiun', 'jayakarta': 'lokasi_stasiun',
    'mgb': 'lokasi_stasiun', 'mangga besar': 'lokasi_stasiun', 'manggabesar': 'lokasi_stasiun',
    'sw': 'lokasi_stasiun', 'sawah besar': 'lokasi_stasiun', 'sawahbesar': 'lokasi_stasiun',
    'jua': 'lokasi_stasiun', 'juanda': 'lokasi_stasiun',
    'gdd': 'lokasi_stasiun', 'gondangdia': 'lokasi_stasiun',
    'cki': 'lokasi_stasiun', 'cikini': 'lokasi_stasiun',
    'mri': 'lokasi_stasiun', 'manggarai': 'lokasi_stasiun',
    'teb': 'lokasi_stasiun', 'tebet': 'lokasi_stasiun',
    'cw': 'lokasi_stasiun', 'cawang': 'lokasi_stasiun',
    'drn': 'lokasi_stasiun', 'duren kalibata': 'lokasi_stasiun', 'durenkalibata': 'lokasi_stasiun', 'durkal': 'lokasi_stasiun',
    'psmb': 'lokasi_stasiun', 'pasar minggu baru': 'lokasi_stasiun', 'pasarminggubaru': 'lokasi_stasiun', 'ps minggu baru': 'lokasi_stasiun', 'pasming baru': 'lokasi_stasiun',
    'psm': 'lokasi_stasiun', 'pasar minggu': 'lokasi_stasiun', 'pasarminggu': 'lokasi_stasiun', 'ps minggu': 'lokasi_stasiun', 'pasming': 'lokasi_stasiun',
    'tnt': 'lokasi_stasiun', 'tanjung barat': 'lokasi_stasiun', 'tanjungbarat': 'lokasi_stasiun', 'tj barat': 'lokasi_stasiun',
    'lna': 'lokasi_stasiun', 'lenteng agung': 'lokasi_stasiun', 'lentengagung': 'lokasi_stasiun', 'lenteng': 'lokasi_stasiun',
    'up': 'lokasi_stasiun', 'universitas pancasila': 'lokasi_stasiun', 'universitaspancasila': 'lokasi_stasiun', 'univ pancasila': 'lokasi_stasiun',
    'ui': 'lokasi_stasiun', 'universitas indonesia': 'lokasi_stasiun', 'universitasindonesia': 'lokasi_stasiun', 'univ indonesia': 'lokasi_stasiun',
    'poc': 'lokasi_stasiun', 'pondok cina': 'lokasi_stasiun', 'pondokcina': 'lokasi_stasiun', 'pocin': 'lokasi_stasiun',
    'dpb': 'lokasi_stasiun', 'depok baru': 'lokasi_stasiun', 'depokbaru': 'lokasi_stasiun', 'detos': 'lokasi_stasiun', 'debar': 'lokasi_stasiun',
    'dp': 'lokasi_stasiun', 'depok': 'lokasi_stasiun', 'depok lama': 'lokasi_stasiun',
    'cta': 'lokasi_stasiun', 'citayam': 'lokasi_stasiun',
    'bjd': 'lokasi_stasiun', 'bojonggede': 'lokasi_stasiun', 'bojong gede': 'lokasi_stasiun',
    'clt': 'lokasi_stasiun', 'cilebut': 'lokasi_stasiun',
    'boo': 'lokasi_stasiun', 'bogor': 'lokasi_stasiun',
    'pdrg': 'lokasi_stasiun', 'pondok rajeg': 'lokasi_stasiun', 'pondokrajeg': 'lokasi_stasiun',
    'cbn': 'lokasi_stasiun', 'cibinong': 'lokasi_stasiun',
    'nmo': 'lokasi_stasiun', 'nambo': 'lokasi_stasiun',

    # LINTAS BEKASI/CIKARANG
    'sudb': 'lokasi_stasiun', 'bni city': 'lokasi_stasiun', 'bnicity': 'lokasi_stasiun', 'sudirman baru': 'lokasi_stasiun',
    'sud': 'lokasi_stasiun', 'sudirman': 'lokasi_stasiun',
    'mtr': 'lokasi_stasiun', 'matraman': 'lokasi_stasiun',
    'jng': 'lokasi_stasiun', 'jatinegara': 'lokasi_stasiun', 'jatneg': 'lokasi_stasiun',
    'kld': 'lokasi_stasiun', 'klender': 'lokasi_stasiun',
    'bua': 'lokasi_stasiun', 'buaran': 'lokasi_stasiun',
    'kldb': 'lokasi_stasiun', 'klender baru': 'lokasi_stasiun', 'klenderbaru': 'lokasi_stasiun',
    'cuk': 'lokasi_stasiun', 'cakung': 'lokasi_stasiun',
    'kri': 'lokasi_stasiun', 'kranji': 'lokasi_stasiun',
    'bks': 'lokasi_stasiun', 'bekasi': 'lokasi_stasiun',
    'bkst': 'lokasi_stasiun', 'bekasi timur': 'lokasi_stasiun', 'bekasitimur': 'lokasi_stasiun', 'bks timur': 'lokasi_stasiun',
    'tb': 'lokasi_stasiun', 'tambun': 'lokasi_stasiun',
    'cit': 'lokasi_stasiun', 'cibitung': 'lokasi_stasiun',
    'tlm': 'lokasi_stasiun', 'metland telagamurni': 'lokasi_stasiun', 'metlandtelagamurni': 'lokasi_stasiun', 'telaga murni': 'lokasi_stasiun',
    'ckr': 'lokasi_stasiun', 'cikarang': 'lokasi_stasiun', 'ckrg': 'lokasi_stasiun',

    # LINTAS RANGKASBITUNG (SERPONG LINE)
    'plm': 'lokasi_stasiun', 'palmerah': 'lokasi_stasiun', 'plmrh': 'lokasi_stasiun',
    'kby': 'lokasi_stasiun', 'kebayoran': 'lokasi_stasiun',
    'pdj': 'lokasi_stasiun', 'pondok ranji': 'lokasi_stasiun', 'pondokranji': 'lokasi_stasiun', 'pd ranji': 'lokasi_stasiun',
    'jmu': 'lokasi_stasiun', 'jurangmangu': 'lokasi_stasiun', 'jurang mangu': 'lokasi_stasiun', 'bxc': 'lokasi_stasiun',
    'sdm': 'lokasi_stasiun', 'sudimara': 'lokasi_stasiun',
    'ru': 'lokasi_stasiun', 'rawa buntu': 'lokasi_stasiun', 'rawabuntu': 'lokasi_stasiun', 'r buntu': 'lokasi_stasiun',
    'srp': 'lokasi_stasiun', 'serpong': 'lokasi_stasiun',
    'csk': 'lokasi_stasiun', 'cisauk': 'lokasi_stasiun',
    'cc': 'lokasi_stasiun', 'cicayur': 'lokasi_stasiun',
    'jtk': 'lokasi_stasiun', 'jatake': 'lokasi_stasiun',
    'prp': 'lokasi_stasiun', 'parungpanjang': 'lokasi_stasiun', 'parung panjang': 'lokasi_stasiun',
    'cjt': 'lokasi_stasiun', 'cilejit': 'lokasi_stasiun',
    'dar': 'lokasi_stasiun', 'daru': 'lokasi_stasiun',
    'tej': 'lokasi_stasiun', 'tenjo': 'lokasi_stasiun',
    'tgs': 'lokasi_stasiun', 'tigaraksa': 'lokasi_stasiun', 'tiga raksa': 'lokasi_stasiun',
    'cky': 'lokasi_stasiun', 'cikoya': 'lokasi_stasiun',
    'mj': 'lokasi_stasiun', 'maja': 'lokasi_stasiun',
    'ctr': 'lokasi_stasiun', 'citeras': 'lokasi_stasiun',
    'rk': 'lokasi_stasiun', 'rangkasbitung': 'lokasi_stasiun', 'rangkas bitung': 'lokasi_stasiun', 'rangkas': 'lokasi_stasiun',

    # LINTAS TANGERANG & LINGKAR KOTA
    'tng': 'lokasi_stasiun', 'tangerang': 'lokasi_stasiun',
    'thi': 'lokasi_stasiun', 'tanah tinggi': 'lokasi_stasiun', 'tanahtinggi': 'lokasi_stasiun',
    'bpr': 'lokasi_stasiun', 'batu ceper': 'lokasi_stasiun', 'batuceper': 'lokasi_stasiun',
    'pi': 'lokasi_stasiun', 'poris': 'lokasi_stasiun',
    'kds': 'lokasi_stasiun', 'kalideres': 'lokasi_stasiun',
    'rw': 'lokasi_stasiun', 'rawa buaya': 'lokasi_stasiun', 'rawabuaya': 'lokasi_stasiun',
    'boi': 'lokasi_stasiun', 'bojong indah': 'lokasi_stasiun', 'bojongindah': 'lokasi_stasiun',
    'tko': 'lokasi_stasiun', 'taman kota': 'lokasi_stasiun', 'tamankota': 'lokasi_stasiun',
    'psg': 'lokasi_stasiun',
    'grg': 'lokasi_stasiun', 'grogol': 'lokasi_stasiun',
    'du': 'lokasi_stasiun', 'duri': 'lokasi_stasiun',
    'ak': 'lokasi_stasiun', 'angke': 'lokasi_stasiun',
    'kpb': 'lokasi_stasiun', 'kampung bandan': 'lokasi_stasiun', 'kampungbandan': 'lokasi_stasiun', 'kp bandan': 'lokasi_stasiun',
    'rjw': 'lokasi_stasiun', 'rajawali': 'lokasi_stasiun',
    'kmo': 'lokasi_stasiun', 'kemayoran': 'lokasi_stasiun',
    'pse': 'lokasi_stasiun', 'pasar senen': 'lokasi_stasiun', 'pasarsenen': 'lokasi_stasiun', 'ps senen': 'lokasi_stasiun', 'senen': 'lokasi_stasiun',
    'gst': 'lokasi_stasiun', 'gang sentiong': 'lokasi_stasiun', 'gangsentiong': 'lokasi_stasiun', 'sentiong': 'lokasi_stasiun',
    'kmt': 'lokasi_stasiun', 'kramat': 'lokasi_stasiun',
    'pok': 'lokasi_stasiun', 'pondok jati': 'lokasi_stasiun', 'pondokjati': 'lokasi_stasiun',
    'thb': 'lokasi_stasiun', 'tanah abang': 'lokasi_stasiun', 'tanahabang': 'lokasi_stasiun', 'tnh abang': 'lokasi_stasiun', 'tn abang': 'lokasi_stasiun',
    'kat': 'lokasi_stasiun', 'karet': 'lokasi_stasiun',
    'ac': 'lokasi_stasiun', 'ancol': 'lokasi_stasiun',
    'tpk': 'lokasi_stasiun', 'tanjung priok': 'lokasi_stasiun', 'tanjungpriok': 'lokasi_stasiun', 'tj priok': 'lokasi_stasiun', 'priok': 'lokasi_stasiun',

    # LINTAS SOLO-JOGJA
    'pl': 'lokasi_stasiun', 'palur': 'lokasi_stasiun',
    'sk': 'lokasi_stasiun', 'solo jebres': 'lokasi_stasiun', 'solojebres': 'lokasi_stasiun',
    'slo': 'lokasi_stasiun', 'solo balapan': 'lokasi_stasiun', 'solobalapan': 'lokasi_stasiun', 'solo': 'lokasi_stasiun',
    'pws': 'lokasi_stasiun', 'purwosari': 'lokasi_stasiun',
    'gw': 'lokasi_stasiun', 'gawok': 'lokasi_stasiun',
    'dl': 'lokasi_stasiun', 'delanggu': 'lokasi_stasiun',
    'ce': 'lokasi_stasiun', 'ceper': 'lokasi_stasiun',
    'kt': 'lokasi_stasiun', 'klaten': 'lokasi_stasiun',
    'swt': 'lokasi_stasiun', 'srowot': 'lokasi_stasiun',
    'bbn': 'lokasi_stasiun', 'brambanan': 'lokasi_stasiun', 'prambanan': 'lokasi_stasiun',
    'mgw': 'lokasi_stasiun', 'maguwo': 'lokasi_stasiun',
    'lpn': 'lokasi_stasiun', 'lempuyangan': 'lokasi_stasiun',
    'yk': 'lokasi_stasiun', 'yogyakarta': 'lokasi_stasiun', 'tugu': 'lokasi_stasiun', 'stasiun tugu': 'lokasi_stasiun',

    # BANDARA
    'bst': 'lokasi_stasiun', 'bandara soekarno hatta': 'lokasi_stasiun', 'bandarasoekarnohatta': 'lokasi_stasiun',
    'bandara soetta': 'lokasi_stasiun', 'soetta': 'lokasi_stasiun'
}

In [ ]:
# Kamus Slang: Standarisasi ke KATA DASAR (Post-Stemming Style)
slang_dict = {
    # Istilah Penumpang KRL (Anker)
    'anker': 'anak kereta', 'krl': 'kereta', 'commuter': 'kereta',
    'pnp': 'tumpang', 'tap': 'tempel', 'gate': 'pintu', 'gerbong': 'kereta', 'ka': 'kereta',
    'loko': 'lokomotif', 'nyangkut': 'tahan', 'ngetem': 'henti', 'oper': 'pindah',
    'transit': 'pindah', 'cl': 'kereta', 'roker': 'rombongan', 'jalur': 'jalur', 'st': 'stasiun',

    # Slang Umum & Kata Berimbuhan (Sudah di-stemming ke Kata Dasar)
    'abal-abal': 'palsu', 'abang': 'kakak', 'abangnya': 'kakak', 'adem': 'tenang',
    'adek': 'adik', 'adek-adek': 'adik', 'adeknya': 'adik', 'ajib': 'hebat',
    'ala-ala': 'seperti', 'amat': 'sangat', 'ambekan': 'marah', 'amin': 'amin',
    'amien': 'amin', 'anak-anaknya': 'anak', 'anjay': 'astaga', 'anjir': 'astaga', 'anu': 'itu',
    'apaan': 'apa', 'apasih': 'apa', 'apik': 'bagus', 'asik': 'asyik',
    'asu': 'anjing', 'atuh': 'saja', 'awak': 'saya', 'awal-awal': 'awal',
    'bacoti': 'bicara', 'bagus-bagus': 'bagus', 'bahas-bahas': 'bahas', 'baju-baju': 'baju',
    'bakal': 'akan', 'bakalan': 'akan', 'bang': 'kakak', 'banget': 'sangat',
    'banget-banget': 'sangat', 'bantuannya': 'bantu', 'banyak-banyak': 'banyak', 'bapakmu': 'ayah',
    'bapaknya': 'ayah', 'bareng': 'sama', 'bareng-bareng': 'sama', 'baru-baru': 'baru',
    'bego': 'bodoh', 'beginian': 'ini', 'begituan': 'itu', 'begitu-gituan': 'itu',
    'bengkak-bengkak': 'bengkak', 'berantem': 'hantam', 'berani-berani': 'berani', 'beda-beda': 'beda',
    'bikin': 'buat', 'boleh-boleh': 'boleh', 'bukti-bukti': 'bukti', 'bunga-bunga': 'bunga', 'buruan': 'cepat', 'buset': 'astaga',
    'cabe-cabean': 'centil', 'cakep': 'tampan', 'cantik-cantik': 'cantik', 'cantik-cantiknya': 'cantik', 'cape': 'lelah', 'capek': 'lelah',
    'cari-cari': 'cari', 'cek-cek': 'cek', 'ceklis': 'centang', 'cemen': 'takut', 'cepatan': 'cepat',
    'cewek': 'perempuan', 'cinggirnya': 'kelingking', 'cowok': 'laki', 'cuman': 'hanya',
    'curhat-curhat': 'curhat', 'cuy': 'kawan', 'dah': 'sudah', 'dalam-dalam': 'dalam',
    'darimana': 'dari mana', 'deh': 'saja', 'dek': 'adik', 'dekat-dekati': 'dekat',
    'demikian': 'begitu', 'diapa-apai': 'apa', 'dibawah': 'bawah', 'dibegitukan': 'laku',
    'dibikin': 'buat', 'dibikin-bikin': 'buat', 'dibelakang': 'belakang', 'dih': 'aduh',
    'dijakarta': 'jakarta', 'dikata-katai': 'kata', 'dikasih': 'beri', 'diluar': 'luar',
    'diluaran': 'luar', 'dimana': 'mana', 'dimana-mana': 'mana', 'dimau': 'mau',
    'dirumah': 'rumah', 'disana': 'sana', 'disini': 'sini', 'disisi': 'sisi',
    'disitu': 'situ', 'doa-doa': 'doa', 'doang': 'saja', 'duitnya': 'uang',
    'duh': 'aduh', 'dulunya': 'dahulu', 'duren': 'durian', 'edan': 'gila',
    'elus-elus': 'elus', 'emak': 'ibu', 'emak-emak': 'ibu', 'embuh': 'tahu',
    'encek': 'paman', 'enggak': 'tidak', 'gak': 'tidak', 'enggak akan': 'tidak akan', 'enggak bakal': 'tidak akan',
    'enggak jelas': 'tidak jelas', 'enggak kuat': 'tidak kuat', 'enggak mau': 'tidak mau', 'enggak pernah': 'tidak pernah',
    'enggak punya': 'tidak punya', 'enggak salah': 'tidak salah', 'enggak tau': 'tidak tahu', 'engkoh': 'kakak',
    'entar': 'nanti', 'entot': 'setubuh', 'euy': 'ya', 'fakta-fakta': 'fakta',
    'foto bareng': 'foto sama', 'foto-foto': 'foto', 'foto-fotonya': 'foto', 'gara-gara': 'karena',
    'geblek': 'bodoh', 'gede': 'besar', 'gedek': 'kesal', 'gegara': 'karena',
    'gendut': 'gemuk', 'geng': 'kelompok', 'geulis': 'cantik', 'gih': 'sana',
    'goblok': 'bodoh', 'gosip-gosipan': 'gosip', 'gua': 'saya', 'gue': 'saya',  'gw': 'saya', 'gweh': 'saya', 'hal-hal': 'hal',
    'hapal': 'hafal', 'harusny': 'harus', 'hepi-hepi': 'senang', 'hotnya': 'seksi',
    'indak': 'tidak', 'iseng-iseng': 'iseng', 'itumah': 'itu', 'iya': 'ya',
    'jaman': 'zaman', 'jaman-jaman': 'zaman', 'jaman-jamannya': 'zaman', 'jari-jari': 'jari',
    'jelek-jelek': 'jelek', 'jidat': 'dahi', 'jomblo': 'lajang', 'kagak': 'tidak',
    'kaget': 'kejut', 'kalo': 'kalau', 'kalo enggak': 'kalau tidak', 'kangen': 'rindu',
    'kasih': 'beri', 'kau': 'kamu', 'kawatir': 'khawatir', 'kayak': 'seperti',
    'kayak begini': 'seperti ini', 'kayaknya': 'seperti', 'kayanya': 'seperti', 'kece': 'keren',
    'kece-kece': 'keren', 'kecil-kecil': 'kecil', 'kek': 'seperti', 'kekeh': 'ngotot',
    'keluar': 'keluar', 'kemana': 'mana', 'kemana-mana': 'mana', 'kenyataan-kenyataan': 'nyata',
    'kepo': 'tahu', 'kepoi': 'tahu', 'kerudungan': 'kerudung', 'kesini': 'sini',
    'kesitu': 'situ', 'ketawa': 'tawa', 'ketemu': 'temu', 'klop': 'cocok',
    'kocak': 'lucu', 'kok': 'mengapa', 'komen': 'komentar', 'komen-komen': 'komentar',
    'komen-komenan': 'komentar', 'komen-komennya': 'komentar', 'komentari': 'komentar', 'komentarnya': 'komentar',
    'komunitas-komunitas': 'komunitas', 'koplak': 'lucu', 'ku': 'aku', 'laginya': 'lagi',
    'lagu-lagunya': 'lagu', 'laki': 'laki', 'lama-lama': 'lama', 'laku-laku': 'laku',
    'lebay': 'lebih', 'ledeki': 'ledek', 'lenjeh': 'genit', 'lihat-lihat': 'lihat',
    'lo': 'kamu', 'lo-lo': 'kamu', 'lu': 'kamu', 'lucu banget': 'lucu sekali',
    'lucu-lucu': 'lucu', 'maafi': 'maaf', 'mah': 'saja', 'mak': 'ibu',
    'makin': 'makin', 'maki-maki': 'maki', 'maknya': 'ibu', 'mampus': 'mati',
    'mangkanya': 'maka', 'mantab': 'mantap', 'marah-marah': 'marah', 'marah-marahan': 'marah',
    'mas': 'kakak', 'mbak': 'kakak', 'mbak-mbak': 'perempuan', 'mbaknya': 'kakak',
    'meleot': 'lemas', 'melulu': 'terus', 'memikir': 'pikir', 'mending': 'baik',
    'mengepoi': 'tahu', 'mengomong': 'bicara', 'mengomongkan': 'bicara', 'mengomongnya': 'bicara',
    'merit': 'nikah', 'mikir': 'pikir', 'miring-miring': 'miring', 'moga': 'semoga',
    'muka': 'wajah', 'muka-muka': 'wajah', 'mukanya': 'wajah', 'nama-nama': 'nama',
    'napsu': 'nafsu', 'neko-neko': 'macam', 'nenek-nenek': 'nenek', 'ngeri': 'ngeri',
    'ngeri-ngeri': 'ngeri', 'nggak': 'tidak', 'ngomong-ngomong': 'bicara', 'nih-nih': 'ini',
    'numpuk': 'tumpuk', 'oma-oma': 'nenek', 'omong doang': 'bicara saja', 'ongkir': 'ongkos',
    'palingan': 'paling', 'pengin': 'ingin', 'penginnya': 'ingin', 'perbulan': 'bulan',
    'pikir-pikir': 'pikir', 'pol': 'sangat', 'pukuli': 'pukul', 'rame-rame': 'ramai',
    'rasai': 'rasa', 'ribet': 'rumit', 'ribet-ribet': 'rumit', 'ridho': 'rida',
    'salihah': 'saleha', 'sampah habis': 'sampah', 'sapa-sapa': 'sapa',
    'say': 'sayang', 'sebelum-sebelumnya': 'sebelum', 'sedikit-sedikit': 'sedikit', 'seindonesia': 'indonesia',
    'seksi-seksi': 'seksi', 'selow': 'santai', 'sempak': 'celana', 'senyumi': 'senyum',
    'seram-seram': 'seram', 'sering-sering': 'sering', 'seru-seru': 'seru', 'setrong': 'kuat',
    'siapa-siapa': 'siapa', 'sirik': 'iri', 'sista-sista': 'saudara', 'sok tau': 'sok tahu',
    'sono': 'sana', 'staf-stafnya': 'staf', 'sukur-sukur': 'syukur', 'tanda-tandanya': 'tanda',
    'tanya-tanya': 'tanya', 'tapi': 'tetapi', 'tau': 'tahu', 'tau tuh': 'tahu itu',
    'tau-tau': 'tahu', 'taunya': 'tahu', 'teh': 'kakak', 'terong-terongan': 'centil',
    'terus-terusan': 'terus', 'teteh': 'kakak', 'tipe-tipe': 'tipe', 'tolol': 'bodoh',
    'tuh': 'itu', 'tumpah-tumpah': 'tumpah', 'tuanya': 'tua', 'ultah': 'ulang tahun',
    'ultahnya': 'ulang tahun', 'unyu': 'lucu', 'unyu-unyu': 'lucu', 'ustad': 'ustaz',
    'uwak': 'paman', 'walah': 'wah', 'woy': 'halo', 'yaallah': 'allah',
    'yakali': 'ya kali', 'yoi': 'ya', 'yuk': 'ayo', 'pansi': 'apa sih', 'kelen': 'kalian', 'satset': 'cepat',
    'wong': 'orang', 'jir': 'astaga', 'mls': 'malas', 'bloon': 'bodoh', 'bengkalai': 'bangkai',
    'ngegendong': 'gendong', 'dibutuhin': 'butuh', 'jalaaan': 'jalan', 'balikin': 'balik', 'lift2': 'lift',
    'tumpuk': 'tumpuk', 'gerbong': 'kereta', 'st': 'stasiun', 'desak2an': 'desak', 'desak2': 'desak',
    'kira2': 'kira', 'bapak2': 'ayah', 'gb': 'kereta', 'tu': 'itu', 'sd': 'sampai dengan', 'jauh2': 'jauh',
    'pinggir': 'pinggir', 'stgh': 'setengah', 'mlm': 'malam', 'lgs': 'langsung', 'didepok': 'di depok', 'tgr': 'tangerang',
    'hmpr': 'hampir'
}

In [ ]:
# Daftar kata negasi yang harus dipertahankan (tidak dihapus sebagai stopword)
negation_terms = {
    'tidak', 'belum', 'tanpa', 'namun', 'tetapi',
    'melainkan', 'walau', 'seharusnya', 'sebetulnya', 'kurang'
}

# Stopword tambahan berdasarkan hasil analisis EDA
custom_stopwords = {
    # Interaction noise (umum pada layanan pelanggan)
    'min', 'admin', 'halo', 'mohon', 'tolong', 'info',
    'terima', 'kasih', 'tanya', 'cek', 'balas', 'respon',
    'terimakasih', 'makasih', 'apa', 'kapan'

    # Fillers & adverbs
    'banget', 'sangat', 'sekali', 'deh', 'sih', 'kok', 'lah', 'nya',
    'si', 'tadi', 'barusan', 'hari', 'kali',
}

# Whitelist untuk mempertahankan token penting hasil normalisasi
whitelist = {'angka', 'waktu_jam', 'rentang_angka', 'lokasi_stasiun'}

## Preprocessing Configuration
berisi parameter dan pengaturan yang digunakan dalam proses pembersihan dan normalisasi teks.

In [ ]:
# Inisialisasi Stopword
# Ambil stopword dasar dari Sastrawi
stopword_factory = StopWordRemoverFactory()
base_stopwords = set(stopword_factory.get_stop_words())

# Gabungkan stopword dasar dengan stopword domain, lalu keluarkan kata negasi dan token whitelist agar tetap dipertahankan
final_stopword_set = (base_stopwords | custom_stopwords) - (negation_terms | whitelist)

In [ ]:
# Inisialisasi stemmer Sastrawi
stemmer_factory = StemmerFactory()                               # membuat factory stemmer
stemmer         = stemmer_factory.create_stemmer()               # membuat objek stemmer

In [ ]:
# Tokenizer NLTK (Hanya mengambil karakter alphanumeric)
tokenizer = RegexpTokenizer(r'\w+')

# Preprocessing

In [ ]:
def map_tokens_with_dictionary(text: str, mapping_dict: dict) -> str:
  """Memetakan token berdasarkan kamus tertentu. Untuk normalisasi berbasis aturan (misalnya slang atau kode stasiun)"""
  tokens = text.split()  # memecah teks menjadi token berbasis spasi

  # mengganti token jika ditemukan dalam kamus, jika tidak tetap
  mapped_tokens = [mapping_dict.get(token, token) for token in tokens]

  return ' '.join(mapped_tokens)  # menggabungkan kembali menjadi teks


In [ ]:
def clean_text_noise(text: str) -> str:
  """Membersihkan noise dasar, case folding, dan normalisasi slang/elongation."""
  # 1. Penghapusan elemen tidak relevan (URL, mention, hashtag)
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'#', '', text)

  # Hapus tanda baca/simbol (tetapi biarkan karakter alfanumerik dan spasi)
  text = re.sub(r'[^\w\s]', ' ', text)

  # 2. Case folding
  text = text.lower()

  # 3. Normalisasi bentuk kata (Elongation & Slang/Penulisan Singkat)
  text = replace_word_elongation(text)  # contoh: "laamaaa" → "lama"
  text = replace_slang(text)            # contoh: "yg" → "yang"

  # Rapikan spasi berlebih setelah semua pembersihan
  text = re.sub(r'\s+', ' ', text).strip()

  return text

In [ ]:
def normalize_text_entities(text: str) -> str:
  """Standarisasi entitas waktu, angka, dan nama stasiun. Untuk engurangi variasi fitur agar model lebih fokus pada pola teks."""
  # 1. Normalisasi ekspresi numerik
  text = re.sub(r'\b\d{1,2}[.:]\d{2}\b', ' waktu_jam ', text)   # format jam
  text = re.sub(r'\b\d+\s*-\s*\d+\b', ' rentang_angka ', text)  # rentang angka
  text = re.sub(r'\b\d+\b', ' angka ', text)                    # angka tunggal

  # 2. Normalisasi bahasa tidak baku/kode stasiun menggunakan kamus
  text = map_tokens_with_dictionary(text, slang_dict)
  text = map_tokens_with_dictionary(text, KAMUS_STASIUN)

  # Rapikan spasi yang mungkin timbul akibat replace
  text = re.sub(r'\s+', ' ', text).strip()

  return text

In [ ]:
def handle_missing_and_duplicates(df: pd.DataFrame,
                                 text_col: str = 'full_text') -> pd.DataFrame:
  """Penanganan data yang hilang atau terduplikasi."""
  # Kondisi awal dataset
  initial_count      = len(df)                                   # jumlah baris sebelum penanganan
  initial_missing    = df[text_col].isna().sum()                 # jumlah nilai kosong pada kolom teks
  initial_duplicates = df.duplicated(subset=[text_col]).sum()    # jumlah data duplikat berdasarkan kolom teks

  print(f'\n[Sebelum Penanganan]')
  print(f'  Jumlah data keseluruhan : {initial_count}')
  print(f'  Nilai kosong ({text_col}) : {initial_missing}')
  print(f'  Data duplikat           : {initial_duplicates}')

  # Penanganan nilai kosong
  df = df.dropna(subset=[text_col])
  after_missing = df[text_col].isna().sum()                      # verifikasi setelah penghapusan

  # Penanganan data duplikat
  df = df.drop_duplicates(subset=[text_col])
  df = df.reset_index(drop=True)
  after_duplicates = df.duplicated(subset=[text_col]).sum()      # verifikasi setelah penghapusan

  # Kondisi akhir dataset
  final_count = len(df)                                          # jumlah baris setelah penanganan

  print(f'\n[Setelah Penanganan]')
  print(f'  Nilai kosong ({text_col}) : {after_missing}')
  print(f'  Data duplikat           : {after_duplicates}')
  print(f'  Jumlah data keseluruhan : {final_count}')
  print(f'\n  Total data dihapus      : {initial_count - final_count} baris')

  return df

In [ ]:
def preprocess_svm(text: str) -> str:
  """Pipeline preprocessing untuk SVM"""
  # 1. Clean noise, case folding, & slang normalization
  text = clean_text_noise(text)

  # 2. Stemming (Sastrawi menerima input string)
  text = stemmer.stem(text)

  # 3. Standarisasi entitas (waktu, angka, stasiun)
  text = normalize_text_entities(text)

  # 4. Tokenization (Menggunakan RegexpTokenizer)
  tokens = tokenizer.tokenize(text)

  # 5. Stopword Removal
  clean_tokens = [word for word in tokens if word not in final_stopword_set]

  # Gabungkan kembali menjadi string untuk di-feed ke TfidfVectorizer
  text = ' '.join(clean_tokens)

  return text

In [ ]:
def preprocess_indobert(text: str) -> str:
  """Pipeline preprocessing untuk IndoBERT"""
  # 1. Clean noise, case folding, & slang normalization
  text = clean_text_noise(text)                                          # terapkan pembersihan noise dasar

  return text

In [ ]:
def evaluate_preprocessing_pipeline(df: pd.DataFrame,
                                    text_col: str = 'full_text',
                                    svm_col: str = 'text_svm',
                                    bert_col: str = 'text_bert',
                                    label_col: str = 'label',
                                  n_samples: int = 2) -> None:
  """
  Fungsi master untuk mengevaluasi hasil preprocessing teks.
  Meliputi preview sampel, statistik reduksi kata, dan quality check.
  """
  # Perbandingan Teks Sebelum dan Sesudah Preprocessing
  print('\n' + '=' * 70)
  print('  [1] PREVIEW PERBANDINGAN TEKS SEBELUM & SESUDAH PREPROCESSING')
  print('=' * 70)

  categories = df[label_col].dropna().unique()

  for label in categories:
      print(f'\n{"─" * 70}')
      print(f'  Kategori: {label.upper()}')
      print(f'{"─" * 70}')
      samples = df[df[label_col] == label].head(n_samples)

      for i, (_, row) in enumerate(samples.iterrows(), 1):
          print(f'\n  Sampel {i}:')
          print(f'  [ASLI]      {row[text_col]}')
          print(f'  [SVM]       {row[svm_col]}')
          print(f'  [IndoBERT]  {row[bert_col]}')

  # Statistik Panjang Teks
  print('\n' + '=' * 70)
  print('  [2] STATISTIK PANJANG TEKS (JUMLAH KATA)')
  print('=' * 70)

  stats = pd.DataFrame({
      'Teks Asli':    df[text_col].apply(lambda x: len(str(x).split())),
      'Setelah SVM':  df[svm_col].apply(lambda x: len(str(x).split())),
      'Setelah BERT': df[bert_col].apply(lambda x: len(str(x).split())),
  })

  print(stats.describe().round(2))
  rata_asli = stats['Teks Asli'].mean()
  rata_svm  = stats['Setelah SVM'].mean()
  if rata_asli > 0:
      reduksi = ((rata_asli - rata_svm) / rata_asli) * 100
      print(f'\n  Rata-rata reduksi jumlah kata setelah preprocessing SVM : {reduksi:.1f}%')
      print(f'  ({rata_asli:.1f} kata → {rata_svm:.1f} kata rata-rata)')

  # Quality Check Data Pasca-Preprocessing
  print('\n' + '=' * 70)
  print('  [3] QUALITY CHECK PASCA-PREPROCESSING')
  print('=' * 70)

  empty_svm  = df[svm_col].apply(lambda x: str(x).strip() == '').sum()
  empty_bert = df[bert_col].apply(lambda x: str(x).strip() == '').sum()
  print(f'\n  Teks kosong setelah preprocessing:')
  print(f'    SVM      : {empty_svm} data')
  print(f'    IndoBERT : {empty_bert} data')

  short_svm = df[svm_col].apply(lambda x: len(str(x).split()) < 3).sum()
  print(f'\n  Teks < 3 kata setelah preprocessing SVM : {short_svm} data')
  print(f'  (Berisiko menghasilkan sparse vector pada TF-IDF)')

  if empty_svm > 0:
      print(f'\n  Contoh teks kosong setelah preprocessing SVM:')
      mask = df[svm_col].apply(lambda x: str(x).strip() == '')
      display(df[mask][[text_col, svm_col, label_col]].head(3))

  if short_svm > 0:
      print(f'\n  Contoh teks < 3 kata setelah preprocessing SVM:')
      mask = df[svm_col].apply(lambda x: len(str(x).split()) > 0 and len(str(x).split()) < 3)
      display(df[mask][[text_col, svm_col, label_col]].head(3))

# Runner

In [ ]:
def run_preprocessing(df: pd.DataFrame,
                      text_col: str = 'full_text',
                    drop_empty: bool = True) -> pd.DataFrame:
  """
  Fungsi master untuk menjalankan seluruh pipeline preprocessing secara berurutan.
  Meliputi pembersihan awal, preprocessing SVM & IndoBERT, evaluasi, dan penyiapan output.
  """
  print('  PIPELINE PREPROCESSING — KAI COMMUTER COMPLAINT')
  print('-' * 70)

  # 1. Penanganan data kotor (Missing values & Duplicates)
  print('\n[Tahap 1] Membersihkan missing values dan duplikat...')
  df = handle_missing_and_duplicates(df, text_col)

  # 2. Preprocessing teks
  print('\n[Tahap 2] Memproses teks untuk model SVM dan IndoBERT...')

  df['text_svm']  = df[text_col].apply(preprocess_svm)
  print('\n[1/2] Preprocessing SVM selesai.')

  df['text_bert'] = df[text_col].apply(preprocess_indobert)
  print('[2/2] Preprocessing IndoBERT selesai.')

  # 3. Evaluasi dan Quality Check
  print('\n[Tahap 3] Menjalankan Evaluasi dan Quality Check...')
  evaluate_preprocessing_pipeline(
        df,
        text_col=text_col,
        svm_col='text_svm',
        bert_col='text_bert',
        label_col='label'
    )

  # 4. Siapkan output
  cols_to_save = [text_col, 'text_svm', 'text_bert', 'label']
  df_output    = df[cols_to_save].copy()


  print(f'\n{'=' * 70}')
  print(f'  RINGKASAN OUTPUT')
  print(f'{'=' * 70}')
  print(f'\n  Total data final : {len(df_output)}')
  print(f'  Kolom tersimpan  : {cols_to_save}')

  print(f'\n  Preview 5 Data Pertama:')
  display(df_output.head())

  return df_output

In [ ]:
# Menjalankan pipeline
df_clean = run_preprocessing(df, text_col='full_text')

  PIPELINE PREPROCESSING — KAI COMMUTER COMPLAINT
----------------------------------------------------------------------

[Tahap 1] Membersihkan missing values dan duplikat...

[Sebelum Penanganan]
  Jumlah data keseluruhan : 4924
  Nilai kosong (full_text) : 0
  Data duplikat           : 6

[Setelah Penanganan]
  Nilai kosong (full_text) : 0
  Data duplikat           : 0
  Jumlah data keseluruhan : 4918

  Total data dihapus      : 6 baris

[Tahap 2] Memproses teks untuk model SVM dan IndoBERT...

[1/2] Preprocessing SVM selesai.
[2/2] Preprocessing IndoBERT selesai.

[Tahap 3] Menjalankan Evaluasi dan Quality Check...

  [1] PREVIEW PERBANDINGAN TEKS SEBELUM & SESUDAH PREPROCESSING

──────────────────────────────────────────────────────────────────────
  Kategori: KEAMANAN
──────────────────────────────────────────────────────────────────────

  Sampel 1:
  [ASLI]      @CommuterLine @suckislife_ @perkeretaapian Ngeri tangganya ga kuat min. Itu banyak bgt manusia
  [SVM]       ngeri t

,full_text,text_svm,label
369,@CommuterLine pepes dah guaaa,pepes,Kepadatan
520,@CommuterLine delay mulu,delay mulu,Keterlambatan
543,@CommuterLine Pantesan numpuk kaya begitu http...,pantesan tumpuk,Kepadatan



  RINGKASAN OUTPUT

  Total data final : 4918
  Kolom tersimpan  : ['full_text', 'text_svm', 'text_bert', 'label']

  Preview 5 Data Pertama:


,full_text,text_svm,text_bert,label
0,@CommuterLine @suckislife_ @perkeretaapian Nge...,ngeri tangga tidak kuat banyak manusia,ngeri tangganya enggak kuat min itu banyak ban...,Keamanan
1,@CommuterLine 5517 telat 50 menit kenapa masuk...,angka telat angka menit masuk lokasi_stasiun a...,5517 telat 50 menit kenapa masuk bks harusnya ...,Keterlambatan
2,@CommuterLine itu papan jadwal cisauk jalur 2 ...,papan jadwal lokasi_stasiun jalur angka tidak ...,itu papan jadwal cisauk jalur 2 enggak update ...,Keterlambatan
3,@CommuterLine @jalur5_ Klo kyk gini petugasnya...,kalau begini tugas bantu astaga mau gendong ka...,kalo kayak begini petugasnya bantu apa anjir m...,Pelayanan
4,@CommuterLine Penataran Malang-SBY pukul jam 3...,natar malang surabaya pukul jam 3an angka des ...,penataran malang surabaya pukul jam 3an 1 des ...,Keterlambatan


In [ ]:
# Export ke CSV
OUTPUT_PATH = '/content/drive/MyDrive/skripsi/svm_indobert/data/final_data/krl_complaints_preprocessed_4.csv'
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f'\nFile disimpan di: {OUTPUT_PATH}')


File disimpan di: /content/drive/MyDrive/skripsi/svm_indobert/data/final_data/krl_complaints_preprocessed_4.csv
